# Hala H22 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â prospective TEST batch replication

One free Kaggle Save & Run All version. No training, no paid compute, no production serving and no change to previous published outcomes. The 2026-10-17 earlier FP16 TEST stays independent; prior INT8 is unknown. The actual assigned T4 is bound on the batch worker before inference. Saved outputs require later retrieval, hashes, independent audits and publication. This notebook is NOT release qualification.


In [ ]:
import os
from pathlib import Path
assert os.name == 'posix' and Path('/kaggle/working').is_dir()
assert os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'
assert not Path('/kaggle/working/hala-h22-batch').exists(), 'Do not overwrite a prior run'
# No model weights downloaded or GPU model allocated in this cell.


Qualify all pinned packages before model allocation. No paid plan, token, service credentials, quotas bypass or auto-upgrade. Installed Torch must already match the reviewed GPU environment.


In [ ]:
%pip install --quiet --no-deps transformers==5.16.1 accelerate==1.12.0 bitsandbytes==0.50.2 peft==0.20.0 safetensors==0.8.0 Pillow==11.3.0
import importlib.metadata
assert importlib.metadata.version('torch') == '2.11.0+cu128'


In [ ]:
# Restore EXACT qualified font from hash-pinned official Ubuntu package.
import hashlib, io, tarfile, urllib.request
from pathlib import Path

PACKAGE_URL = "https://archive.ubuntu.com/ubuntu/pool/main/f/fonts-dejavu/fonts-dejavu-core_2.37-8_all.deb"
PACKAGE_SHA = "40049660c194f3b8a2541fc7369efebb10e9f94bdac836a2f38fafedd10fa73a"
FONT_SHA = "ae7b7855e115a5966d8b1b3f80f254ccc117ec86f9965e202ee2940453837280"
target = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
sha = lambda b: hashlib.sha256(b).hexdigest()
if target.exists():
    assert target.is_file() and not target.is_symlink()
    assert sha(target.read_bytes()) == FONT_SHA, "Existing system font differs from frozen media"
else:
    with urllib.request.urlopen(PACKAGE_URL, timeout=45) as response:
        assert response.geturl() == PACKAGE_URL, "Unexpected archive redirect"
        package = response.read(2 * 1024**2 + 1)
    assert len(package) <= 2 * 1024**2 and sha(package) == PACKAGE_SHA
    assert package.startswith(b"!<arch>\n")
    cursor, found = 8, []
    while cursor + 60 <= len(package):
        hdr = package[cursor:cursor+60]
        size = int(hdr[48:58].strip())
        name = hdr[:16].decode("ascii").strip().rstrip("/")
        assert 0 <= size <= len(package)-cursor-60
        data = package[cursor+60:cursor+60+size]
        if name.startswith("data.tar."):
            with tarfile.open(fileobj=io.BytesIO(data), mode="r:*") as tar:
                for member in tar.getmembers():
                    if member.name.lstrip("./") == "usr/share/fonts/truetype/dejavu/DejaVuSans.ttf":
                        assert member.isfile() and member.size == 759720
                        payload = tar.extractfile(member).read(1024*1024)
                        assert len(payload) == member.size and sha(payload) == FONT_SHA
                        found.append(payload)
        cursor += 60 + size + size % 2
    assert len(found) == 1, "Exactly one pinned font asset required"
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("xb") as output:
        output.write(found[0])
assert sha(target.read_bytes()) == FONT_SHA, "Qualified font unavailable"
print("HALA_H22_PINNED_FONT_QUALIFIED", FONT_SHA)


In [ ]:
import hashlib, json, subprocess, sys
from pathlib import Path
data = "\"\"\"Prospective H22 free-Kaggle Save & Run All batch operator.\r\n\r\nThis operator does not train, tune, merge, deploy or publish model artifacts.\r\nIt is deliberately distinct from the historical interactive attempt3 operator.\r\nAll three TEST modes must run with the same observed physical GPU in ONE batch.\r\n\"\"\"\r\nimport hashlib\r\nimport importlib.metadata\r\nimport io\r\nimport json\r\nimport os\r\nfrom pathlib import Path\r\nimport re\r\nimport shutil\r\nimport subprocess\r\nimport sys\r\nimport time\r\nimport urllib.request\r\nimport zipfile\r\n\r\nSOURCE = \"07591f109b6683c9302fcf890259ddf0bde91494\"\r\nFREEZE = \"98ca7bba6fc16dd6cd737eaad6de1e425e30aef67116c379db36fbbb85b2af4b\"\r\nMODEL_REV = \"89644892e4d85e24eaac8bacfd4f463576704203\"\r\nARTIFACT_COMMIT = \"c7f2e16244910c7b4246749b3186d49fca5ce7b4\"\r\nDEV_SHA = \"60a0986cc196041c2474314624e0f88c9c2986d75bc58cf0ddb44623ed294e02\"\r\nH19_SHA = \"d6be1ffba54a3a44978da4c922130fc2d7e74bdc2532a14d0d1e7cc3a691cead\"\r\nH19_MANIFEST = \"ced87ae6874df6de2ed1ab853312824084191a29a42b97869f6eb462aceb629e\"\r\nADAPTER_MANIFEST = \"7e46995aa694e5782a93d4beb6139cc5880b898a2038eb2d2ce2a2d8006a841d\"\r\nEARLIER_FP16_ARCHIVE = \"e098a7631708aa50609e56ba7701c01a1c5cc5804b87f9994b6dded181abf471\"\r\nEARLIER_FP16_COMMIT = \"9568d58c74f3e1326b55b04d304f643e2bf1b99f\"\r\nARCHIVE_BASE = \"https://raw.githubusercontent.com/TheHalfMoon/Hala/\"\r\nMODES = (\"fp16\", \"int8\", \"nf4\")\r\nROOT = Path(\"/kaggle/working/hala-h22-batch\")\r\nCACHE = \"/tmp/hala-h22-cache\"\r\nADAPTER_FILES = {\r\n    \"adapter_config.json\": \"21bbc4037d9fe27e1281700e72dcc402c4ce801d2d27798ac64cba33ac4d5de2\",\r\n    \"adapter_model.safetensors\": \"7eb2f17df11151e5b77a5c715c59f56c854ca7b0d0da3ee591f394b501826fa7\",\r\n    \"training_metadata.json\": \"d1251b39c3d71fcd410ec0fce631b2c065a6df4978309bedaff168807314020d\",\r\n    \"manifest.json\": ADAPTER_MANIFEST,\r\n}\r\nADAPTER_IDENTITY = {\r\n    \"base_model\": \"Qwen/Qwen3-VL-2B-Instruct\", \"base_revision\": MODEL_REV,\r\n    \"data_sha256\": \"af540e4db2289540fb8adecd0665439c4d6055ffc86673a3286e902c86c0340e\",\r\n    \"evaluation_sha256\": \"e9f98694089d5411d3a6c570c7de2ede8f82cbeb3132e47904d17301030c0768\",\r\n    \"source_commit\": \"5b6eb4bf29acbaf21aa199f113f78af4f268b3b7\",\r\n}\r\nDEV = (\r\n    (\"e252b7805055fa6a89e1d92b6bba7faa5dbf1efc\", \"6d5cfe7c3746a76eaceb8ebc37605d0d98cc69b69ca94765421f081b64d64580\"),\r\n    (\"e3d400424766aad3c5856ba29266e11033392ef5\", \"e0dbe2d692c522943b3e0c49c198e540416cbabfa2b19499c9da4a518b8339e5\"),\r\n    (\"d08f522de433d9da866ec3907161e4a9b326efba\", \"84dac2fd6ab44d751cefbe9ce61a958ef44a39aad3b1f09e880c5350bbd87458\"),\r\n)\r\n\r\n\r\ndef digest(raw):\r\n    return hashlib.sha256(raw).hexdigest()\r\n\r\n\r\ndef atomic_json(path, payload):\r\n    path = Path(path)\r\n    temp = path.with_suffix(path.suffix + \".partial\")\r\n    with temp.open(\"w\", encoding=\"utf-8\") as handle:\r\n        json.dump(payload, handle, sort_keys=True, allow_nan=False)\r\n        handle.write(\"\\n\")\r\n        handle.flush()\r\n        os.fsync(handle.fileno())\r\n    temp.replace(path)\r\n\r\n\r\ndef public_artifact(commit, suffix, expected, maximum):\r\n    \"\"\"Read only immutable public artifacts; bounds and digest are mandatory.\"\"\"\r\n    if re.fullmatch(r\"[a-f0-9]{40}\", commit) is None:\r\n        raise ValueError(\"Immutable commit SHA required\")\r\n    if not suffix or suffix.startswith(\"/\") or \"..\" in Path(suffix).parts:\r\n        raise ValueError(\"Unsafe artifact path\")\r\n    url = ARCHIVE_BASE + commit + \"/\" + suffix\r\n    with urllib.request.urlopen(url, timeout=45) as reply:\r\n        if reply.geturl() != url:\r\n            raise ValueError(\"Unexpected artifact redirect\")\r\n        raw = reply.read(maximum + 1)\r\n    if len(raw) > maximum or digest(raw) != expected:\r\n        raise ValueError(\"Public evidence digest or size mismatch: \" + suffix)\r\n    return raw\r\n\r\n\r\ndef safe_unpack(raw, target):\r\n    \"\"\"No traversal, links, duplicate members, ZIP bombs or replacement.\"\"\"\r\n    target = Path(target)\r\n    if target.exists():\r\n        raise FileExistsError(\"Existing evidence/dataset directory must not be replaced\")\r\n    target.mkdir(parents=True)\r\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\r\n        infos = z.infolist()\r\n        if len(infos) > 512 or len({x.filename for x in infos}) != len(infos):\r\n            raise ValueError(\"Unexpected ZIP member count or duplicates\")\r\n        if sum(x.file_size for x in infos) > 32 * 1024**2:\r\n            raise ValueError(\"ZIP content budget exceeded\")\r\n        for item in infos:\r\n            name = item.filename\r\n            if (not name or name.startswith(\"/\") or chr(92) in name or\r\n                    any(x in (\"\", \".\", \"..\") for x in name.rstrip(\"/\").split(\"/\")) or\r\n                    item.file_size > 8 * 1024**2 or ((item.external_attr >> 16) & 0o170000) == 0o120000):\r\n                raise ValueError(\"Unsafe ZIP member\")\r\n            path = target / name\r\n            if item.is_dir():\r\n                path.mkdir(parents=True, exist_ok=True)\r\n                continue\r\n            path.parent.mkdir(parents=True, exist_ok=True)\r\n            with z.open(item) as src, path.open(\"xb\") as dst:\r\n                shutil.copyfileobj(src, dst, length=1024 * 1024)\r\n\r\n\r\ndef prepare(root=ROOT):\r\n    \"\"\"Network-only source/evidence preparation. No model download.\"\"\"\r\n    if os.name != \"posix\" or not Path(\"/kaggle/working\").is_dir():\r\n        raise RuntimeError(\"Must run in actual remote Kaggle Linux environment\")\r\n    if root.exists():\r\n        raise FileExistsError(\"Batch workspace already exists; never overwrite old evidence\")\r\n    root.mkdir(parents=True)\r\n    source = root / \"source\"\r\n    subprocess.run([\"git\", \"clone\", \"--quiet\", \"https://github.com/TheHalfMoon/Hala.git\", str(source)],\r\n                   check=True, timeout=240)\r\n    subprocess.run([\"git\", \"checkout\", \"--detach\", SOURCE], cwd=source, check=True, timeout=60,\r\n                   stdout=subprocess.DEVNULL)\r\n    if subprocess.check_output([\"git\", \"rev-parse\", \"HEAD\"], cwd=source, text=True).strip() != SOURCE:\r\n        raise ValueError(\"Historical source identity mismatch\")\r\n    if subprocess.check_output([\"git\", \"status\", \"--porcelain\", \"--untracked-files=all\"], cwd=source, text=True).strip():\r\n        raise ValueError(\"Historical source must be clean\")\r\n    safe_unpack(public_artifact(ARTIFACT_COMMIT, \"h19-freeze/hala-h21-frozen.zip\", H19_SHA, 3 * 1024**2),\r\n                root / \"frozen\")\r\n    adapter = root / \"retained-adapter\"\r\n    adapter.mkdir()\r\n    for name, expected in ADAPTER_FILES.items():\r\n        bound = 4 * 1024**2 if name.endswith(\"safetensors\") else 65536\r\n        (adapter / name).write_bytes(public_artifact(ARTIFACT_COMMIT,\r\n            \"h21-shared-pilot/trained-adapter/\" + name, expected, bound))\r\n    sys.path.insert(0, str(source))\r\n    from development.freeze import verify_bundle\r\n    from training.adapter_archive import verify_bundle as verify_adapter\r\n    verify_bundle(root / \"frozen\", require_media=True, expected_manifest_sha256=H19_MANIFEST)\r\n    verify_adapter(adapter, ADAPTER_IDENTITY, ADAPTER_MANIFEST)\r\n    # Immutable external evidence is verified before rerunning old TEST cases.\r\n    for n, (commit, expected) in enumerate(DEV):\r\n        mode = MODES[n]\r\n        raw = public_artifact(commit, f\"h22-attempt2/profile-{n}-{mode}-dev-evidence.zip\",\r\n                              expected, 16 * 1024**2)\r\n        with zipfile.ZipFile(io.BytesIO(raw)) as z:\r\n            if digest(z.read(\"experiment.json\")) != DEV_SHA:\r\n                raise ValueError(\"Retained DEV experiment hash mismatch\")\r\n    previous = public_artifact(EARLIER_FP16_COMMIT,\r\n        \"h22-test-restart/profile-0-fp16-test-evidence.zip\", EARLIER_FP16_ARCHIVE, 16 * 1024**2)\r\n    with zipfile.ZipFile(io.BytesIO(previous)) as z:\r\n        last = json.loads(z.read(\"profile-status.json\"))\r\n        if last.get(\"index\") != 0 or last.get(\"status\") != \"finished\":\r\n            raise ValueError(\"Earlier TEST evidence is not a completed FP16 run\")\r\n    return source\r\n\r\n\r\ndef qualify(source, output):\r\n    os.environ[\"HALA_REMOTE_DEVELOPMENT_ALLOWED\"] = \"1\"\r\n    os.environ[\"HF_HUB_DISABLE_TELEMETRY\"] = \"1\"\r\n    os.environ[\"DO_NOT_TRACK\"] = \"1\"\r\n    for key in (\"HF_HOME\", \"HF_HUB_CACHE\", \"HUGGINGFACE_HUB_CACHE\",\r\n                \"TRANSFORMERS_CACHE\", \"XDG_CACHE_HOME\", \"TORCH_HOME\"):\r\n        if key in os.environ and Path(os.environ[key]).resolve() != Path(CACHE).resolve():\r\n            raise ValueError(\"Unreviewed cache environment override\")\r\n        os.environ[key] = CACHE\r\n    from evaluation import development_precision as precision\r\n    from evaluation.development_runtime import evaluation_guard\r\n    from development.media import FONT_SHA256\r\n    from PIL import features\r\n    evaluation_guard(\"kaggle\")\r\n    precision.verify_sources(FREEZE)\r\n    precision.check_cache(CACHE)\r\n    for package, expected in {\"torch\": \"2.11.0+cu128\", \"transformers\": \"5.16.1\",\r\n             \"Pillow\": \"11.3.0\", \"accelerate\": \"1.12.0\", \"bitsandbytes\": \"0.50.2\",\r\n             \"safetensors\": \"0.8.0\", \"peft\": \"0.20.0\"}.items():\r\n        if importlib.metadata.version(package) != expected:\r\n            raise RuntimeError(\"Unqualified environment package: \" + package)\r\n    import torch\r\n    if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != \"Tesla T4\":\r\n        raise RuntimeError(\"Actual free Tesla T4 required\")\r\n    uuid = precision.resolve_device_uuid(torch)\r\n    if features.version_feature(\"raqm\") != \"0.10.1\":\r\n        raise RuntimeError(\"RAQM rendering mismatch\")\r\n    font = Path(\"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\")\r\n    if digest(font.read_bytes()) != FONT_SHA256:\r\n        raise RuntimeError(\"Font content mismatch\")\r\n    report = {\"schema\": \"hala.h22.batch_assigned_hardware.v1\",\r\n              \"provider\": \"kaggle\", \"kernel_run_type\": os.getenv(\"KAGGLE_KERNEL_RUN_TYPE\"),\r\n              \"gpu_name\": torch.cuda.get_device_name(0), \"gpu_uuid\": uuid,\r\n              \"source_commit\": SOURCE, \"source_freeze_sha256\": FREEZE,\r\n              \"h19_manifest_sha256\": H19_MANIFEST, \"adapter_manifest_sha256\": ADAPTER_MANIFEST,\r\n              \"earlier_fp16_replicated_not_overwritten\": True,\r\n              \"earlier_int8_unknown\": True, \"earlier_nf4_not_started\": True,\r\n              \"batch_profile_order\": list(MODES),\r\n              \"no_adoption\": True, \"training_runs\": 0}\r\n    atomic_json(output / \"assigned-hardware.json\", report)\r\n    return report\r\n\r\n\r\n\r\ndef completion_counts(result):\r\n    \"\"\"Independent schema guard for measured eight-episode bilingual H19 result.\"\"\"\r\n    if type(result) is not dict or result.get(\"status\") != \"measured\":\r\n        raise ValueError(\"Only fully measured results have completion counts\")\r\n    summary = result.get(\"summary\")\r\n    if type(summary) is not dict or not {\"all\", \"en\", \"ar\"}.issubset(summary):\r\n        raise ValueError(\"H19 language summary missing\")\r\n    completed = {}\r\n    for language, total in ((\"all\", 8), (\"en\", 4), (\"ar\", 4)):\r\n        section = summary[language]\r\n        if type(section) is not dict or type(section.get(\"episodes\")) is not int or section[\"episodes\"] != total:\r\n            raise ValueError(\"Unexpected H19 episode count\")\r\n        value = section.get(\"episodes_complete\")\r\n        if type(value) is not int or not 0 <= value <= total:\r\n            raise ValueError(\"Invalid independently verified completion count\")\r\n        completed[language] = value\r\n    if completed[\"all\"] != completed[\"en\"] + completed[\"ar\"]:\r\n        raise ValueError(\"Per-language verified completion counts disagree\")\r\n    return completed\r\n\r\n\r\ndef run(root=ROOT):\r\n    source = prepare(root)\r\n    results = root / \"results\"\r\n    results.mkdir()\r\n    batch = {\"schema\": \"hala.h22.batch_test_triplet.v1\",\r\n             \"status\": \"preparing\", \"source_commit\": SOURCE,\r\n             \"earlier_fp16_archive_sha256\": EARLIER_FP16_ARCHIVE,\r\n             \"earlier_int8_outcome\": \"unknown\", \"earlier_nf4_outcome\": \"not_started\",\r\n             \"profiles\": [], \"training_runs\": 0, \"model_adopted\": False}\r\n    atomic_json(results / \"batch-status.json\", batch)\r\n    assigned = qualify(source, results)\r\n    batch[\"assigned_hardware\"] = assigned\r\n    batch[\"status\"] = \"running\"\r\n    atomic_json(results / \"batch-status.json\", batch)\r\n    env = dict(os.environ, PYTHONIOENCODING=\"utf-8\", CUBLAS_WORKSPACE_CONFIG=\":4096:8\")\r\n    for idx, mode in enumerate(MODES):\r\n        if subprocess.check_output([\"git\", \"status\", \"--porcelain\", \"--untracked-files=all\"],\r\n                                   cwd=source, text=True).strip():\r\n            raise RuntimeError(\"Scientific source changed during batch\")\r\n        if subprocess.check_output([\"git\", \"rev-parse\", \"HEAD\"], cwd=source, text=True).strip() != SOURCE:\r\n            raise RuntimeError(\"Scientific source commit changed during batch\")\r\n        # Observe the physical UUID again before every new process allocation.\r\n        import torch\r\n        from evaluation.development_precision import resolve_device_uuid\r\n        if resolve_device_uuid(torch) != assigned[\"gpu_uuid\"]:\r\n            raise RuntimeError(\"Physical GPU changed during TEST batch\")\r\n        name = f\"profile-{idx}-{mode}-test\"\r\n        folder = results / name\r\n        folder.mkdir()\r\n        args = [sys.executable, \"-u\", \"-m\", \"evaluation.development_precision\",\r\n                \"--mode\", mode, \"--source-freeze-sha256\", FREEZE,\r\n                \"--bundle\", str(root / \"frozen\"),\r\n                \"--manifest-sha256\", H19_MANIFEST, \"--output\", str(folder / \"run\"),\r\n                \"--font-path\", \"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\",\r\n                \"--source-commit\", SOURCE, \"--provider\", \"kaggle\", \"--split\", \"test\",\r\n                \"--adapter\", str(root / \"retained-adapter\"),\r\n                \"--adapter-manifest-sha256\", ADAPTER_MANIFEST,\r\n                \"--adapter-identity\", json.dumps(ADAPTER_IDENTITY, sort_keys=True)]\r\n        record = {\"mode\": mode, \"split\": \"test\", \"status\": \"running\",\r\n                  \"started_unix\": time.time(), \"gpu_uuid\": assigned[\"gpu_uuid\"]}\r\n        batch[\"profiles\"].append(record)\r\n        atomic_json(results / \"batch-status.json\", batch)\r\n        try:\r\n            with (folder / \"profile.log\").open(\"wb\") as log:\r\n                process = subprocess.run(args, cwd=source, env=env, stdout=log,\r\n                                         stderr=subprocess.STDOUT, timeout=3700)\r\n            record[\"status\"] = \"finished\" if process.returncode == 0 else \"failed\"\r\n            record[\"returncode\"] = process.returncode\r\n        except subprocess.TimeoutExpired:\r\n            record[\"status\"] = \"timeout\"\r\n        except Exception as error:\r\n            record[\"status\"] = \"launcher_error\"\r\n            record[\"error_type\"] = type(error).__name__\r\n        result = folder / \"run\" / \"result.json\"\r\n        if result.is_file():\r\n            try:\r\n                content = result.read_bytes()\r\n                record[\"result_sha256\"] = digest(content)\r\n                measured = json.loads(content)\r\n                record[\"result_status\"] = measured.get(\"status\")\r\n                record[\"completed_by_language\"] = completion_counts(measured)\r\n                if measured.get(\"gpu_uuid\") != assigned[\"gpu_uuid\"]:\r\n                    record[\"status\"] = \"physical_device_mismatch\"\r\n            except (OSError, ValueError, TypeError, KeyError) as error:\r\n                # Malformed/incomplete GPU output must NOT suppress archival of\r\n                # the raw file, stdout, exit status, and previous profiles.\r\n                record[\"status\"] = \"invalid_result\"\r\n                record[\"result_record_error\"] = type(error).__name__\r\n        record[\"finished_unix\"] = time.time()\r\n        atomic_json(folder / \"profile-status.json\", record)\r\n        archive = results / (name + \"-evidence.zip\")\r\n        with zipfile.ZipFile(archive, \"x\", compression=zipfile.ZIP_DEFLATED) as z:\r\n            for path in sorted(folder.rglob(\"*\")):\r\n                if path.is_file():\r\n                    if path.is_symlink() or path.stat().st_size > 8 * 1024**2:\r\n                        raise ValueError(\"Evidence file exceeds bound; preserve raw directory\")\r\n                    z.write(path, path.relative_to(folder).as_posix())\r\n            z.write(results / \"assigned-hardware.json\", \"assigned-hardware.json\")\r\n            z.write(Path(__file__), \"h22_batch_operator.py\")\r\n        if archive.stat().st_size > 16 * 1024**2:\r\n            raise ValueError(\"Evidence ZIP exceeds limit; raw directory preserved\")\r\n        record[\"archive_sha256\"] = digest(archive.read_bytes())\r\n        record[\"archive_bytes\"] = archive.stat().st_size\r\n        atomic_json(results / \"batch-status.json\", batch)\r\n        print(\"HALA_BATCH_PROFILE:\", json.dumps(record, sort_keys=True), flush=True)\r\n    batch[\"status\"] = (\"completed_unqualified\" if all(\r\n        x[\"status\"] == \"finished\" and x.get(\"result_status\") == \"measured\"\r\n        for x in batch[\"profiles\"]) else \"completed_with_blockers\")\r\n    atomic_json(results / \"batch-status.json\", batch)\r\n    print(\"HALA_BATCH_FINISHED:\", json.dumps(batch, sort_keys=True), flush=True)\r\n    return batch\r\n\r\n\r\nif __name__ == \"__main__\":\r\n    run()\r\n"
expected = "67c1f31b32f449ddf9d5f84e75ab406b224f33c82943ed8a99d1a8721174f952"
assert hashlib.sha256(data.encode('utf-8')).hexdigest() == expected
path = Path('/kaggle/working/hala-h22-operator.py')
assert not path.exists(), 'Do not overwrite existing operator source'
path.write_bytes(data.encode('utf-8'))
print('OPERATOR_SHA256', expected)
subprocess.run([sys.executable, '-u', str(path)], check=True,
               timeout=3*3700+1800)
print('NO PUBLIC RESULT CLAIM UNTIL VERSION OUTPUT RETRIEVAL AND INDEPENDENT AUDIT')
